# 00. Verify the data sources (PetrIA-Ingest)

**Purpose.** The research report is a list of *claims* about the official
datasets. This notebook checks each claim against the real portal and the real
files, and prints one report.

**How to run (Google Colab).** Runtime → Run all.

**Run 1 lesson (2026-11-20).** The catalog (`datos.gob.ar`) answered, but the
file host (`datos.energia.gob.ar`) did not answer from Colab. So there are two
ways to get the files:
- **A. Download from Colab** (works if the file host answers).
- **B. Upload the files you downloaded yourself** (section 3). This is the path
  to use now. The list of files and links is in `notebooks/README.md`.

**Run 3 (targeted checks).** Section 7 answers the questions run 2 left open:
the newest month of data, swapped coordinates, placeholder zeros, repeated well
names, formation spellings, concession codes, and how many wells could train the
detector. Files uploaded earlier in the same Colab session are reused.

**What it does not do.** It writes nothing to any database. It never invents a
value: every failure goes into `errors` in the report.

**What to send back.** The last cell prints the full report. Copy it all.

In [ ]:
%pip -q install pyogrio pyproj requests pandas

In [ ]:
import codecs
import hashlib
import json
import re
import time
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

WORKDIR = Path("data_verification")
WORKDIR.mkdir(exist_ok=True)
HEADERS = {"User-Agent": "petria-verification-notebook/1.2"}
MAX_DOWNLOAD_MB = 400
TIMEOUT = (8, 30)        # (connect, read) seconds: do not wait long on a silent host
UNREACHABLE = set()      # hosts that did not answer: we stop trying them
UPLOAD_FILES = True      # True: you upload files you downloaded yourself (section 3)

# Portals to try, in order. Run 1 confirmed datos.gob.ar runs CKAN.
HOSTS = ["https://datos.gob.ar", "https://datos.energia.gob.ar"]

# Candidate dataset slugs. Run 1 confirmed the first of each pair; the second
# slug (older name) returned 404 on datos.gob.ar.
CANDIDATE_SLUGS = {
    "production_chapter_iv": [
        "energia-produccion-petroleo-gas-por-pozo-capitulo-iv",
        "produccion-de-petroleo-y-gas-por-pozo",
    ],
    "fracturing_adjunto_iv": [
        "energia-datos-fractura-pozos-hidrocarburos-adjunto-iv",
        "datos-de-fractura-de-pozos-adjunto-iv",
    ],
    "concessions": [
        "energia-produccion-hidrocarburos---concesiones-explotacion",
        "energia-produccion-hidrocarburos---lotes-explotacion",
        "energia-exploracion-hidrocarburos---permisos-exploracion",
    ],
}
SEARCH_QUERIES = [
    "capitulo iv pozos",
    "fractura adjunto iv",
    "concesiones explotacion",
    "lotes explotacion",
    "permisos exploracion",
]

REPORT = {
    "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "errors": [],
    "hosts_unreachable": [],
    "datasets": {},
    "search": {},
    "uploads": {},
    "downloads": {},
    "profiles": {},
    "joins": {},
    "claims_check": {},
}

## 1. Ask the portal what it really has
CKAN is the software behind the portal. Its API tells us, for each dataset, the
license, the last modification and the list of files. `ckan()` retries
temporary failures (500, read timeouts), gives up at once on a 404, and stops
waiting on a host that does not answer at all.

In [ ]:
def log_error(where, exc):
    REPORT["errors"].append({"where": where, "error": f"{type(exc).__name__}: {exc}"[:300]})


def mark_unreachable(url, exc):
    host = urlparse(url).netloc
    if host not in UNREACHABLE:
        UNREACHABLE.add(host)
        REPORT["hosts_unreachable"].append({"host": host, "error": f"{type(exc).__name__}: {exc}"[:200]})


def ckan(action, **params):
    """Call a CKAN action on each reachable host: (host, result) or (None, None)."""
    for host in HOSTS:
        if urlparse(host).netloc in UNREACHABLE:
            continue
        for attempt in range(3):
            try:
                response = requests.get(
                    f"{host}/api/3/action/{action}", params=params, headers=HEADERS, timeout=TIMEOUT
                )
                response.raise_for_status()
                body = response.json()
                if body.get("success"):
                    return host, body["result"]
                log_error(f"{action} {params} @ {host}", RuntimeError("success is false"))
                break
            except requests.exceptions.ConnectionError as exc:  # includes connect timeouts
                mark_unreachable(host, exc)
                break
            except Exception as exc:
                log_error(f"{action} {params} @ {host} (attempt {attempt + 1})", exc)
                status = getattr(getattr(exc, "response", None), "status_code", None)
                if status is not None and 400 <= status < 500:
                    break  # a 404 will not change on retry
                time.sleep(2 * (attempt + 1))
    return None, None


def summarize_package(pkg):
    return {
        "title": pkg.get("title"),
        "license_id": pkg.get("license_id"),
        "license_title": pkg.get("license_title"),
        "metadata_modified": pkg.get("metadata_modified"),
        "extras": {e.get("key"): e.get("value") for e in pkg.get("extras", [])},
        "resources": [
            {
                "id": r.get("id"),
                "name": r.get("name"),
                "format": r.get("format"),
                "url": r.get("url"),
                "size": r.get("size"),
                "last_modified": r.get("last_modified") or r.get("metadata_modified"),
            }
            for r in pkg.get("resources", [])
        ],
    }

In [ ]:
for label, slugs in CANDIDATE_SLUGS.items():
    REPORT["datasets"][label] = {}
    for slug in slugs:
        host, pkg = ckan("package_show", id=slug)
        REPORT["datasets"][label][slug] = (
            {"host": host, **summarize_package(pkg)} if pkg else {"found": False}
        )

for query in SEARCH_QUERIES:
    host, result = ckan("package_search", q=query, rows=8)
    REPORT["search"][query] = (
        [
            {
                "name": p.get("name"),
                "title": p.get("title"),
                "license_title": p.get("license_title"),
                "metadata_modified": p.get("metadata_modified"),
            }
            for p in result.get("results", [])
        ]
        if result
        else []
    )

rows = [
    {
        "dataset": label,
        "slug": slug,
        "found": info.get("found", True),
        "license": info.get("license_title"),
        "modified": info.get("metadata_modified"),
        "n_resources": len(info.get("resources", [])),
    }
    for label, by_slug in REPORT["datasets"].items()
    for slug, info in by_slug.items()
]
pd.DataFrame(rows)

## 2. Choose which files to inspect
We pick resources by name, per dataset. When several files share a name (run 1
found two for 2024 and two for 2025), we take the most recently modified one.
**Check the table this cell prints.**

In [ ]:
resources = [
    {"dataset": label, "slug": slug, **r}
    for label, by_slug in REPORT["datasets"].items()
    for slug, info in by_slug.items()
    for r in info.get("resources", [])
]

# key: (dataset label, regex on the resource name, accepted formats)
PICK = {
    "registry_csv": ("production_chapter_iv", r"cap[ií]tulo iv\s*-\s*pozos", {"csv"}),
    "wells_shapefile": ("production_chapter_iv", r"cap[ií]tulo iv\s*-\s*pozos", {"shp", "zip"}),
    "padron_csv": ("production_chapter_iv", r"padr[oó]n de pozos", {"csv"}),
    "wells_listing_csv": ("production_chapter_iv", r"listado de pozos", {"csv"}),
    "production_unconventional_csv": ("production_chapter_iv", r"no convencional", {"csv"}),
    "fracturing_csv": ("fracturing_adjunto_iv", r"fractura", {"csv"}),
    "concessions_shp": ("concessions", r"concesi", {"zip", "shp"}),
    "lots_shp": ("concessions", r"lotes", {"zip", "shp"}),
    "permits_shp": ("concessions", r"permisos", {"zip", "shp"}),
}
ZIP_KEYS = {"wells_shapefile", "concessions_shp", "lots_shp", "permits_shp"}
OVERRIDE_URLS = {}


def local_path(key):
    return WORKDIR / f"{key}{'.zip' if key in ZIP_KEYS else '.csv'}"


def pick_resources():
    picked = {}
    for key, (label, pattern, formats) in PICK.items():
        matches = [
            r
            for r in resources
            if r["dataset"] == label
            and re.search(pattern, f"{r.get('name') or ''} {r['slug']}", re.I)
            and (r.get("format") or "").lower() in formats
        ]
        if matches:
            picked[key] = max(matches, key=lambda r: r.get("last_modified") or "")
    for key, url in OVERRIDE_URLS.items():
        picked[key] = {"name": "override", "format": Path(url).suffix.lstrip("."), "url": url, "size": None}
    return picked


picked = pick_resources()
print("NOT picked:", sorted(set(PICK) - set(picked)) or "none")
pd.DataFrame(
    [
        {"key": k, "resource": r.get("name"), "format": r.get("format"), "modified": r.get("last_modified"), "url": r.get("url")}
        for k, r in picked.items()
    ]
)

## 3. Upload the files you downloaded yourself (path B)
Run 1 showed that `datos.energia.gob.ar`, the host that serves the files, did
not answer from Colab. If you downloaded the files on your own device (list in
`notebooks/README.md`), run this cell and pick them. Names are matched by
pattern, so a browser adding `(1)` to the name is fine. Set `UPLOAD_FILES =
False` in the config cell to skip this step.

In [ ]:
FILENAME_KEYS = [  # regex on the file name the portal gives (seen in the resource URLs)
    ("registry_csv", r"capitulo-iv-pozos"),
    ("padron_csv", r"padr.?n-de-pozos"),
    ("wells_listing_csv", r"listado-de-pozos"),
    ("production_unconventional_csv", r"no-convencional"),
    ("fracturing_csv", r"fractura"),
    ("wells_shapefile", r"shapefile-de-pozos"),
    ("concessions_shp", r"concesiones-de-explotaci"),
    ("lots_shp", r"lotes"),
    ("permits_shp", r"permisos"),
]


def key_for(filename):
    for key, pattern in FILENAME_KEYS:
        if re.search(pattern, filename, re.I):
            return key
    return None


if UPLOAD_FILES:
    from google.colab import files

    for name, content in files.upload().items():
        key = key_for(name)
        expected = ".zip" if key in ZIP_KEYS else ".csv"
        if key is None or Path(name).suffix.lower() != expected:
            log_error(f"upload {name}", RuntimeError(f"name not recognized, or not a {expected} file"))
            print(f"SKIPPED {name!r}: name not recognized, or not a {expected} file")
            continue
        local_path(key).write_bytes(content)
        REPORT["uploads"][key] = {
            "original_name": name,
            "bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
        }
        picked.setdefault(key, {"name": name, "format": expected[1:], "url": None, "size": len(content)})
print("uploaded:", sorted(REPORT["uploads"]))

## 4. The tools: download, profile a CSV, inspect a shapefile
Three small functions. `profile_csv` reads the file in chunks, so memory stays
low. It counts rows and nulls, lists the real distinct values of the formation
and resource columns, checks the coordinates, and looks for duplicate
(`idpozo`, `anio`, `mes`) keys. `inspect_geo` reads the real coordinate system
from the `.prj` file and the real attribute names.

In [ ]:
def download(url, dest, max_mb=MAX_DOWNLOAD_MB):
    """Stream a file to disk, hash it, and stop if it is too large."""
    sha, size = hashlib.sha256(), 0
    with requests.get(url, headers=HEADERS, stream=True, timeout=(8, 60), allow_redirects=True) as response:
        response.raise_for_status()
        with open(dest, "wb") as handle:
            for chunk in response.iter_content(1 << 20):
                size += len(chunk)
                if size > max_mb * (1 << 20):
                    raise RuntimeError(f"file is larger than {max_mb} MB; stopped")
                sha.update(chunk)
                handle.write(chunk)
        return {
            "path": str(dest),
            "bytes": size,
            "sha256": sha.hexdigest(),
            "content_type": response.headers.get("content-type"),
            "final_url": response.url,
        }


CATEGORICAL = {"formacion", "formprod", "tipo_de_recurso", "sub_tipo_recurso", "cuenca",
               "provincia", "tipoestado", "tipopozo", "clasificacion", "subclasificacion"}
COORDS = ("coordenadax", "coordenaday")


def sniff(path):
    """Guess the encoding and the delimiter from the first bytes."""
    raw = Path(path).read_bytes()[:4096]
    text, encoding = "", "latin-1"
    for candidate in ("utf-8-sig", "latin-1"):
        try:
            text = codecs.getincrementaldecoder(candidate)().decode(raw, final=False)
            encoding = candidate
            break
        except UnicodeDecodeError:
            continue
    first = text.splitlines()[0] if text else ""
    return encoding, max([",", ";", "\t", "|"], key=first.count)


def profile_csv(path, chunksize=200_000, top=15):
    """Return (profile, key_sets). Everything is read as text: no silent type guessing."""
    encoding, sep = sniff(path)
    header = pd.read_csv(path, sep=sep, encoding=encoding, nrows=0).columns.tolist()
    lower = {c.lower(): c for c in header}
    n_rows, nulls = 0, Counter()
    distinct = {lower[c]: Counter() for c in CATEGORICAL if c in lower}
    coord = {lower[c]: {"min": None, "max": None, "null": 0, "zero": 0} for c in COORDS if c in lower}
    keys = {name: set() for name in ("idpozo", "sigla") if name in lower}
    composite = [lower[c] for c in ("idpozo", "anio", "mes") if c in lower]
    seen, duplicate_rows = set(), 0

    for chunk in pd.read_csv(path, sep=sep, encoding=encoding, dtype=str, chunksize=chunksize, low_memory=False):
        n_rows += len(chunk)
        nulls.update({k: int(v) for k, v in chunk.isna().sum().items()})
        for col, counter in distinct.items():
            counter.update(chunk[col].fillna("<NA>").value_counts().to_dict())
        for col, stats in coord.items():
            values = pd.to_numeric(chunk[col].str.replace(",", ".", regex=False), errors="coerce")
            stats["null"] += int(values.isna().sum())
            stats["zero"] += int((values == 0).sum())
            if values.notna().any():
                low, high = float(values.min()), float(values.max())
                stats["min"] = low if stats["min"] is None else min(stats["min"], low)
                stats["max"] = high if stats["max"] is None else max(stats["max"], high)
        for name, store in keys.items():
            store.update(chunk[lower[name]].dropna().unique())
        if len(composite) == 3:
            for row in zip(*(chunk[c] for c in composite)):
                if row in seen:
                    duplicate_rows += 1
                else:
                    seen.add(row)

    profile = {
        "encoding": encoding,
        "separator": sep,
        "n_rows": n_rows,
        "n_columns": len(header),
        "columns": header,
        "null_counts": {k: v for k, v in nulls.items() if v},
        "distinct_values": {
            col: {"n_distinct": len(c), "top": dict(c.most_common(top))} for col, c in distinct.items()
        },
        "coordinate_stats": coord,
        "n_unique_keys": {name: len(store) for name, store in keys.items()},
        "duplicate_idpozo_anio_mes_rows": duplicate_rows if len(composite) == 3 else "not computed",
        "sample_rows": pd.read_csv(path, sep=sep, encoding=encoding, dtype=str, nrows=3).to_dict("records"),
    }
    return profile, keys


def inspect_geo(path):
    """Read the real CRS, geometry type, feature count and attribute names of a zipped shapefile."""
    from pyogrio import list_layers, read_dataframe, read_info
    from pyproj import CRS

    out = {"prj_files": {}, "layers": {}}
    with zipfile.ZipFile(path) as zf:
        for name in zf.namelist():
            if name.lower().endswith(".prj"):
                text = zf.read(name).decode("utf-8", "replace")
                try:
                    epsg = CRS.from_wkt(text).to_epsg()
                except Exception:
                    epsg = None
                out["prj_files"][name] = {"epsg_guess": epsg, "wkt_start": text[:160]}
    for layer_name, _ in list_layers(path):
        info = read_info(path, layer=layer_name)
        sample = read_dataframe(path, layer=layer_name, max_features=3)
        bounds = info.get("total_bounds")
        out["layers"][layer_name] = {
            "geometry_type": info.get("geometry_type"),
            "features": int(info["features"]),
            "crs": info.get("crs"),
            "fields": list(info["fields"]),
            "bounds": [float(x) for x in bounds] if bounds is not None else None,
            "sample": sample.drop(columns="geometry").astype(str).to_dict("records"),
        }
    return out

## 5. Inspect the files
Uploaded files are used as they are. Anything not uploaded is downloaded, unless
its host already failed to answer.

In [ ]:
KEY_SETS = {}
for key, r in picked.items():
    dest = local_path(key)
    try:
        if key in REPORT["uploads"]:
            REPORT["downloads"][key] = {"source": "upload", **REPORT["uploads"][key]}
        elif dest.exists():  # uploaded earlier in this same session: no need to upload again
            REPORT["downloads"][key] = {
                "source": "local file from an earlier upload",
                "bytes": dest.stat().st_size,
                "sha256": hashlib.sha256(dest.read_bytes()).hexdigest(),
            }
        else:
            if not r.get("url"):
                raise RuntimeError("no URL to download")
            host = urlparse(r["url"]).netloc
            if host in UNREACHABLE:
                raise RuntimeError(f"{host} did not answer earlier: upload this file instead (section 3)")
            REPORT["downloads"][key] = {"source": "download", **download(r["url"], dest)}
        if dest.suffix == ".csv":
            REPORT["profiles"][key], KEY_SETS[key] = profile_csv(dest)
        else:
            REPORT["profiles"][key] = inspect_geo(dest)
    except Exception as exc:
        log_error(f"download or profile: {key}", exc)
        if isinstance(exc, requests.exceptions.ConnectionError) and r.get("url"):
            mark_unreachable(r["url"], exc)

print("profiled:", sorted(REPORT["profiles"]))
print("errors so far:", len(REPORT["errors"]))

## 6. Do the datasets join? And do the research claims hold?
Two checks. **Joins:** how many wells of one file appear in another, and how many
rows per well. **Claims:** the columns the research expected versus the real header.

In [ ]:
def overlap(a, b):
    both = len(a & b)
    return {
        "n_a": len(a),
        "n_b": len(b),
        "in_both": both,
        "pct_a_found_in_b": round(100 * both / len(a), 2) if a else None,
        "pct_b_found_in_a": round(100 * both / len(b), 2) if b else None,
    }


PAIRS = [
    ("fracturing_csv", "registry_csv"),
    ("fracturing_csv", "production_unconventional_csv"),
    ("production_unconventional_csv", "registry_csv"),
    ("padron_csv", "registry_csv"),
    ("fracturing_csv", "padron_csv"),
]
for a, b in PAIRS:
    name = f"{a}.idpozo vs {b}.idpozo"
    if "idpozo" in KEY_SETS.get(a, {}) and "idpozo" in KEY_SETS.get(b, {}):
        REPORT["joins"][name] = overlap(KEY_SETS[a]["idpozo"], KEY_SETS[b]["idpozo"])
    else:
        REPORT["joins"][name] = "not computed: file or column missing"

registry = REPORT["profiles"].get("registry_csv")
if registry:
    REPORT["joins"]["registry rows vs unique idpozo vs unique sigla"] = {
        "rows": registry["n_rows"],
        "unique_idpozo": registry["n_unique_keys"].get("idpozo"),
        "unique_sigla": registry["n_unique_keys"].get("sigla"),
    }
frac = REPORT["profiles"].get("fracturing_csv")
if frac:
    REPORT["joins"]["fracturing rows vs unique idpozo"] = {
        "rows": frac["n_rows"],
        "unique_idpozo": frac["n_unique_keys"].get("idpozo"),
    }

# Claims copied from the research report (UNVERIFIED until compared here).
CLAIMS = {
    "registry_csv": ["idpozo", "sigla", "formprod", "coordenadax", "coordenaday"],
    "fracturing_csv": [
        "idpozo", "longitud_rama_horizontal_m", "cantidad_fracturas",
        "arena_bombeada_nacional_tn", "arena_bombeada_importada_tn",
        "co2_inyectado_m3", "duracion_fractura_dias",
    ],
    "production_unconventional_csv": [
        "idpozo", "sigla", "anio", "mes", "prod_pet", "prod_gas", "prod_agua", "tef",
        "formprod", "formacion", "cuenca", "provincia", "tipo_de_recurso",
        "sub_tipo_recurso", "rectificado", "habilitado", "coordenadax", "coordenaday",
    ],
}
for key, expected in CLAIMS.items():
    profile = REPORT["profiles"].get(key)
    if not profile:
        REPORT["claims_check"][key] = "no profile (file not available)"
        continue
    actual = [c.lower() for c in profile["columns"]]
    REPORT["claims_check"][key] = {
        "n_columns": len(actual),
        "expected_but_missing": sorted(set(expected) - set(actual)),
        "present_but_not_in_claims": sorted(set(actual) - set(expected)),
    }
REPORT["claims_check"]

## 7. Targeted checks (run 3)
Each check reads only the columns it needs and stores its result under
`checks` in the report. A check whose file is missing says so and is skipped.
One failing check never stops the others.

In [ ]:
BOX = {"lon": (-74.0, -53.0), "lat": (-56.0, -21.0)}  # a generous box around Argentina's wells
CHECKS = REPORT.setdefault("checks", {})


def clean(obj):
    """Make numpy numbers and NaN safe for JSON."""
    return json.loads(json.dumps(obj, default=lambda o: o.item() if hasattr(o, "item") else str(o)))


def run_check(name, needs, fn):
    missing = [k for k in needs if not local_path(k).exists()]
    if missing:
        CHECKS[name] = f"skipped: file not available: {missing}"
        return
    try:
        CHECKS[name] = clean(fn())
    except Exception as exc:
        log_error(f"check {name}", exc)


def read_cols(key, cols):
    path = local_path(key)
    encoding, sep = sniff(path)
    return pd.read_csv(path, sep=sep, encoding=encoding, dtype=str, usecols=lambda c: c in cols, low_memory=False)


def norm_text(series):
    return series.fillna("").str.strip().str.lower()


def to_num(series):
    """Numbers from text, accepting a decimal comma. Anything unparseable becomes NaN (and is counted)."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    return pd.to_numeric(series.str.replace(",", ".", regex=False), errors="coerce")


def period(idx):
    return f"{int(idx) // 12}-{int(idx) % 12 + 1:02d}"


def audit_points(lon, lat, labels):
    """Count points outside the Argentine box, and how many a longitude/latitude swap would fix."""
    lon, lat = to_num(lon), to_num(lat)
    present = lon.notna() & lat.notna()
    inside = lon.between(*BOX["lon"]) & lat.between(*BOX["lat"])
    swapped_inside = lat.between(*BOX["lon"]) & lon.between(*BOX["lat"])
    outside = present & ~inside
    sample = labels[outside].head(5).copy()
    sample["lon"], sample["lat"] = lon[outside].head(5), lat[outside].head(5)
    return {
        "n_points": int(len(lon)),
        "n_missing": int((~present).sum()),
        "n_inside_box": int((present & inside).sum()),
        "n_outside_box": int(outside.sum()),
        "n_fixed_by_swap": int((outside & swapped_inside).sum()),
        "n_not_fixed_by_swap": int((outside & ~swapped_inside).sum()),
        "sample_outside": sample.to_dict("records"),
    }


def point_xy(text):
    try:
        lon, lat = json.loads(text)["coordinates"][:2]
        return float(lon), float(lat)
    except Exception:
        return float("nan"), float("nan")


def check_registry_coords():
    df = read_cols("registry_csv", ["idpozo", "sigla", "geojson"])
    xy = df["geojson"].map(point_xy)
    return audit_points(xy.map(lambda t: t[0]), xy.map(lambda t: t[1]), df[["idpozo", "sigla"]])


def check_production_coords():
    df = read_cols("production_unconventional_csv", ["idpozo", "sigla", "coordenadax", "coordenaday"])
    df["coordenadax"] = df["coordenadax"].str.replace(",", ".", regex=False)
    df["coordenaday"] = df["coordenaday"].str.replace(",", ".", regex=False)
    changing = df.groupby("idpozo")[["coordenadax", "coordenaday"]].nunique().max(axis=1)
    wells = df.drop_duplicates("idpozo")
    out = audit_points(wells["coordenadax"], wells["coordenaday"], wells[["idpozo", "sigla"]])
    out["n_wells_with_changing_coordinates"] = int((changing > 1).sum())
    return out


def check_shapefile_coords():
    from pyogrio import read_dataframe

    gdf = read_dataframe(local_path("wells_shapefile"))
    return audit_points(gdf.geometry.x, gdf.geometry.y, gdf[["IDENTIFICA", "SIGLA"]])


def check_periods():
    out = {}
    prod = read_cols("production_unconventional_csv", ["anio", "mes", "formacion"])
    prod["idx"] = pd.to_numeric(prod["anio"]) * 12 + pd.to_numeric(prod["mes"]) - 1
    for label, sub in (("all_rows", prod), ("vaca_muerta_rows", prod[norm_text(prod["formacion"]).eq("vaca muerta")])):
        counts = sub["idx"].value_counts().sort_index()
        out[label] = {
            "first_period": period(sub["idx"].min()),
            "last_period": period(sub["idx"].max()),
            "rows_per_year": sub["anio"].value_counts().sort_index().to_dict(),
            "rows_in_last_6_periods": {period(i): int(n) for i, n in counts.tail(6).items()},
        }
    if local_path("padron_csv").exists():
        pad = read_cols("padron_csv", ["idpozo", "anio", "mes"])
        pad["idx"] = pd.to_numeric(pad["anio"]) * 12 + pd.to_numeric(pad["mes"]) - 1
        first_month = int((pad["idx"] == 2006 * 12).sum())
        out["padron"] = {
            "first_period": period(pad["idx"].min()),
            "last_period": period(pad["idx"].max()),
            "wells_with_first_period_2006_01": first_month,
            "share_with_first_period_2006_01": round(first_month / len(pad), 4),
            "wells_per_first_year": pad["anio"].value_counts().sort_index().to_dict(),
        }
    if local_path("fracturing_csv").exists():
        fr = read_cols("fracturing_csv", ["fecha_inicio_fractura", "fecha_fin_fractura", "fecha_data", "anio_if"])
        out["fracturing"] = {
            "first_start": str(pd.to_datetime(fr["fecha_inicio_fractura"], errors="coerce").min()),
            "last_start": str(pd.to_datetime(fr["fecha_inicio_fractura"], errors="coerce").max()),
            "last_end": str(pd.to_datetime(fr["fecha_fin_fractura"], errors="coerce").max()),
            "last_load_date": str(pd.to_datetime(fr["fecha_data"], errors="coerce").max()),
            "operations_per_start_year": fr["anio_if"].value_counts().sort_index().to_dict(),
        }
    return out


def check_fracturing_zeros():
    numeric = ["longitud_rama_horizontal_m", "cantidad_fracturas", "arena_bombeada_nacional_tn",
               "arena_bombeada_importada_tn", "agua_inyectada_m3", "presion_maxima_psi", "potencia_equipos_fractura_hp"]
    text = ["formacion_productiva", "tipo_reservorio", "subtipo_reservorio", "tipo_terminacion"]
    df = read_cols("fracturing_csv", ["idpozo"] + text + numeric)
    for c in numeric:
        df[c] = to_num(df[c])
    vm = df[norm_text(df["formacion_productiva"]).eq("vaca muerta")]

    def rates(sub):
        return {c: {"zero": int((sub[c] == 0).sum()), "empty": int(sub[c].isna().sum()), "of": int(len(sub))} for c in numeric}

    sand = vm["arena_bombeada_nacional_tn"].fillna(0) + vm["arena_bombeada_importada_tn"].fillna(0)
    return {
        "all_rows": rates(df),
        "vaca_muerta_rows": rates(vm),
        "vaca_muerta_lateral_zero_but_stages_positive": int(((vm["longitud_rama_horizontal_m"] == 0) & (vm["cantidad_fracturas"] > 0)).sum()),
        "vaca_muerta_total_sand_zero": int((sand == 0).sum()),
        "vaca_muerta_summary": vm[numeric].describe(percentiles=[0.1, 0.5, 0.9]).round(2).to_dict(),
        "distinct_values": {c: df[c].value_counts(dropna=False).head(15).to_dict() for c in text},
    }


def check_repeated_sigla():
    df = read_cols("registry_csv", ["idpozo", "sigla", "formacion", "geojson"])
    g = df.groupby("sigla").agg(
        n_ids=("idpozo", "nunique"),
        n_formacion=("formacion", lambda s: s.nunique(dropna=False)),
        n_locations=("geojson", "nunique"),
    )
    several = g[g["n_ids"] > 1]
    return {
        "n_sigla": int(len(g)),
        "n_sigla_with_several_ids": int(len(several)),
        "extra_ids": int((several["n_ids"] - 1).sum()),
        "several_ids_one_formacion": int((several["n_formacion"] == 1).sum()),
        "several_ids_different_formacion": int((several["n_formacion"] > 1).sum()),
        "several_ids_one_location": int((several["n_locations"] == 1).sum()),
        "ids_per_sigla": several["n_ids"].value_counts().sort_index().to_dict(),
        "sample": several.head(5).reset_index().to_dict("records"),
    }


def check_formacion_values():
    prod = read_cols("production_unconventional_csv", ["formacion", "formprod"])
    reg = read_cols("registry_csv", ["formacion"])
    variants = reg[reg["formacion"].fillna("").str.contains("vaca|quintuco", case=False, regex=True)]
    return {
        "production_formacion_all_values": prod["formacion"].value_counts(dropna=False).to_dict(),
        "production_formprod_all_codes": prod["formprod"].value_counts(dropna=False).to_dict(),
        "registry_vaca_or_quintuco_spellings": variants["formacion"].value_counts().to_dict(),
    }


def check_flags_and_numbers():
    df = read_cols("production_unconventional_csv", ["rectificado", "habilitado", "observaciones", "prod_pet", "prod_gas", "prod_agua", "tef"])
    unparsed = {c: int(df[c].notna().sum() - to_num(df[c]).notna().sum()) for c in ("prod_pet", "prod_gas", "prod_agua", "tef")}
    for c in ("prod_pet", "prod_gas", "prod_agua", "tef"):
        df[c] = to_num(df[c])
    return {
        "values_not_parsed_as_numbers": unparsed,
        "rectificado": df["rectificado"].value_counts(dropna=False).to_dict(),
        "habilitado": df["habilitado"].value_counts(dropna=False).to_dict(),
        "observaciones_top": df["observaciones"].value_counts().head(8).to_dict(),
        "numeric_summary": df[["prod_pet", "prod_gas", "prod_agua", "tef"]].describe(percentiles=[0.5, 0.9, 0.99]).round(3).to_dict(),
        "tef_over_31": int((df["tef"] > 31).sum()),
        "negative_values": {c: int((df[c] < 0).sum()) for c in ("prod_pet", "prod_gas", "prod_agua")},
    }


def check_concession_keys():
    from pyogrio import read_dataframe

    def norm(series):
        return {str(x).strip().upper() for x in series.dropna()}

    conc = read_dataframe(local_path("concessions_shp"))
    codes, names = norm(conc["CODIGO_DE_"]), norm(conc["NOMBRE_DE_"])
    reg = read_cols("registry_csv", ["cod_area", "cod_yacimiento", "area", "yacimiento"])
    prod = read_cols("production_unconventional_csv", ["idareapermisoconcesion", "areapermisoconcesion", "idareayacimiento", "areayacimiento"])
    pairs = {
        "concession codes vs registry.cod_area": (codes, norm(reg["cod_area"])),
        "concession codes vs registry.cod_yacimiento": (codes, norm(reg["cod_yacimiento"])),
        "concession names vs registry.area": (names, norm(reg["area"])),
        "concession names vs registry.yacimiento": (names, norm(reg["yacimiento"])),
        "concession codes vs production.idareapermisoconcesion": (codes, norm(prod["idareapermisoconcesion"])),
        "concession codes vs production.idareayacimiento": (codes, norm(prod["idareayacimiento"])),
        "concession names vs production.areapermisoconcesion": (names, norm(prod["areapermisoconcesion"])),
        "concession names vs production.areayacimiento": (names, norm(prod["areayacimiento"])),
    }
    return {name: overlap(a, b) for name, (a, b) in pairs.items()}


def check_target_viability():
    """How many Vaca Muerta wells could train the detector (12-month cumulative oil)?"""
    prod = read_cols("production_unconventional_csv", ["idpozo", "anio", "mes", "prod_pet", "formacion", "tipopozo"])
    vm = prod[norm_text(prod["formacion"]).eq("vaca muerta")].copy()
    vm["idx"] = pd.to_numeric(vm["anio"]) * 12 + pd.to_numeric(vm["mes"]) - 1
    vm["oil"] = to_num(vm["prod_pet"])
    vm = vm.drop_duplicates(["idpozo", "idx"])
    last = int(vm["idx"].max())
    first = vm.groupby("idpozo")["idx"].min()
    window = vm.join(first.rename("first"), on="idpozo")
    window = window[window["idx"] - window["first"] < 12]
    months = window.groupby("idpozo")["idx"].nunique()
    full_ids = set(months[months == 12].index)
    could_be_full = set(first[first + 11 <= last].index)
    not_censored = set(first[first > 2006 * 12].index)
    cum = window[window["idpozo"].isin(full_ids)].groupby("idpozo")["oil"].sum()
    usable = cum[cum.index.isin(not_censored)]
    out = {
        "last_period_in_vaca_muerta_rows": period(last),
        "n_vaca_muerta_wells": int(len(first)),
        "n_oil_values_not_parsed": int(vm["prod_pet"].notna().sum() - vm["oil"].notna().sum()),
        "n_with_12_complete_months": len(full_ids),
        "n_window_open_possible_but_has_gaps": len(could_be_full - full_ids),
        "n_first_period_2006_01": int((first == 2006 * 12).sum()),
        "n_usable_complete_and_not_cut_off": int(len(usable)),
        "cumulative_oil_12m_usable_summary": usable.describe(percentiles=[0.1, 0.5, 0.9]).round(2).to_dict(),
        "share_usable_with_zero_oil": round(float((usable == 0).mean()), 4) if len(usable) else None,
        "well_type_of_vaca_muerta_wells": vm.drop_duplicates("idpozo")["tipopozo"].value_counts(dropna=False).to_dict(),
    }
    if local_path("fracturing_csv").exists():
        frac_ids = set(read_cols("fracturing_csv", ["idpozo"])["idpozo"].dropna())
        out["n_usable_with_a_fracturing_record"] = int(len(set(usable.index) & frac_ids))
    return out


run_check("registry_coordinates", ["registry_csv"], check_registry_coords)
run_check("production_coordinates", ["production_unconventional_csv"], check_production_coords)
run_check("shapefile_coordinates", ["wells_shapefile"], check_shapefile_coords)
run_check("periods_and_freshness", ["production_unconventional_csv"], check_periods)
run_check("fracturing_zeros", ["fracturing_csv"], check_fracturing_zeros)
run_check("repeated_sigla", ["registry_csv"], check_repeated_sigla)
run_check("formacion_values", ["production_unconventional_csv", "registry_csv"], check_formacion_values)
run_check("flags_and_numbers", ["production_unconventional_csv"], check_flags_and_numbers)
run_check("concession_keys", ["concessions_shp", "registry_csv", "production_unconventional_csv"], check_concession_keys)
run_check("target_viability", ["production_unconventional_csv"], check_target_viability)
print("checks done:", {k: ("skipped" if isinstance(v, str) else "ok") for k, v in CHECKS.items()})


## 8. The report to paste back
Copy the whole output of this cell into the chat. It is also saved as
`verification_report.json` in the Colab files panel.

In [ ]:
REPORT["finished_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
text = json.dumps(REPORT, indent=1, ensure_ascii=False, default=str)
Path("verification_report.json").write_text(text, encoding="utf-8")
print(text)